# Notebook 8 — Formas y objetos con Pillow

**Curso:** EITC/AI/GVAPI — European IT Certification, Google Vision API
**Lección 8:** Understanding Shapes and Objects (Drawing Object Borders Using Pillow Python Library)

La Vision API te da números: cuadros delimitadores y vértices. Para *ver* lo que detectó, necesitas dibujar esos números sobre la imagen — y ahí entra **Pillow** (heredera de la histórica librería PIL, Python Imaging Library), la librería estándar de Python para abrir, dibujar sobre y guardar imágenes.

In [ ]:
import os
# os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "ruta/a/tu/clave.json"

from google.cloud import vision
from PIL import Image, ImageDraw, ImageFont

client = vision.ImageAnnotatorClient()
print("Cliente listo.")


## El flujo general

1. La Vision API detecta objetos y te devuelve coordenadas (`bounding_poly.vertices` o `normalized_vertices`)
2. Pillow abre la imagen y crea un objeto de dibujo (`ImageDraw.Draw`) sobre ella
3. Pillow traza rectángulos, líneas o polígonos usando esas coordenadas
4. Opcionalmente, Pillow añade una etiqueta de texto junto a cada objeto
5. Pillow guarda o muestra la imagen resultante

## Dibujar un cuadro delimitador simple

In [ ]:
def dibujar_bbox_simple(image_path, object_bbox, output_path):
    image = Image.open(image_path)
    draw = ImageDraw.Draw(image)
    draw.rectangle(object_bbox, outline="red", width=2)
    image.save(output_path)
    return output_path


# dibujar_bbox_simple("imagen.jpg", (50, 50, 200, 200), "imagen_anotada.jpg")


## Función `draw_vertices`: unir vértices y cerrar el polígono

Cuando trabajas con `object_localization` o `landmark_detection`, la Vision API te da una lista de vértices (no solo dos esquinas), así que conviene una función que una cada vértice con el siguiente **y cierre el contorno**.

In [ ]:
from io import BytesIO

def draw_vertices(image_source, vertices, display_text="", color="green", width=4):
    if isinstance(image_source, bytes):
        pillow_img = Image.open(BytesIO(image_source))
    else:
        pillow_img = Image.open(image_source)

    draw = ImageDraw.Draw(pillow_img)

    for i in range(len(vertices) - 1):
        draw.line(
            [(vertices[i][0], vertices[i][1]), (vertices[i + 1][0], vertices[i + 1][1])],
            fill=color,
            width=width,
        )

    # Línea de cierre: del último vértice al primero
    draw.line(
        [(vertices[-1][0], vertices[-1][1]), (vertices[0][0], vertices[0][1])],
        fill=color,
        width=width,
    )

    if display_text:
        font = ImageFont.load_default()
        draw.text((vertices[0][0] + 10, vertices[0][1]), display_text, font=font, fill="white")

    return pillow_img


## Ejercicio 1 — Dibuja los cuadros delimitadores de una detección real

Completa `anotar_objetos_detectados` para que:

1. Llame a `client.object_localization()` sobre una imagen real
2. Convierta los `normalized_vertices` de cada objeto a coordenadas de píxel (multiplicando por el ancho y el alto reales de la imagen)
3. Dibuje el polígono de cada objeto con `draw_vertices`, usando `obj.name` como `display_text`
4. Guarde la imagen final anotada

In [ ]:
def anotar_objetos_detectados(image_path, output_path):
    pil_image = Image.open(image_path)
    width, height = pil_image.size

    with open(image_path, "rb") as image_file:
        content = image_file.read()

    response = client.object_localization(image=vision.Image(content=content))

    imagen_actual = pil_image
    for obj in response.localized_object_annotations:
        # TODO 1: convierte normalized_vertices a coordenadas de píxel (x * width, y * height)
        vertices_px = ...

        # TODO 2: dibuja el polígono sobre imagen_actual usando draw_vertices,
        # con f"{obj.name} ({obj.score:.2f})" como display_text
        imagen_actual = ...

    imagen_actual.save(output_path)
    return output_path


# anotar_objetos_detectados("salon.jpg", "salon_anotado.jpg")


<details>
<summary><strong>Ver solución</strong></summary>

```python
def anotar_objetos_detectados(image_path, output_path):
    pil_image = Image.open(image_path)
    width, height = pil_image.size

    with open(image_path, "rb") as image_file:
        content = image_file.read()

    response = client.object_localization(image=vision.Image(content=content))

    imagen_actual = pil_image
    for obj in response.localized_object_annotations:
        vertices_px = [(v.x * width, v.y * height) for v in obj.bounding_poly.normalized_vertices]
        imagen_actual = draw_vertices(
            imagen_actual, vertices_px, display_text=f"{obj.name} ({obj.score:.2f})"
        )

    imagen_actual.save(output_path)
    return output_path


anotar_objetos_detectados("salon.jpg", "salon_anotado.jpg")
```

</details>

## Ejercicio 2 — `draw.polygon()` frente al bucle de líneas

`ImageDraw` incluye un método `draw.polygon(...)` que dibuja un polígono cerrado directamente a partir de una lista de puntos, sin que tengas que escribir tú el bucle de líneas de `draw_vertices`.

Completa `comparar_dibujo` para que dibuje el mismo conjunto de vértices dos veces sobre copias distintas de la misma imagen: una con `draw_vertices` (línea por línea) y otra con `draw.polygon()` directamente, y guarde ambos resultados para que puedas compararlos visualmente.

In [ ]:
def comparar_dibujo(image_path, vertices, salida_lineas, salida_polygon):
    # TODO 1: usa draw_vertices sobre una copia de la imagen y guarda en salida_lineas
    imagen_lineas = ...
    imagen_lineas.save(salida_lineas)

    # TODO 2: usa ImageDraw.Draw(...).polygon(vertices, outline="green") sobre otra copia
    # y guarda en salida_polygon
    imagen_polygon = Image.open(image_path).copy()
    draw = ImageDraw.Draw(imagen_polygon)
    ...
    imagen_polygon.save(salida_polygon)

    return salida_lineas, salida_polygon


# comparar_dibujo("test.jpg", [(100, 100), (200, 100), (200, 200), (100, 200)], "con_lineas.jpg", "con_polygon.jpg")


<details>
<summary><strong>Ver solución</strong></summary>

```python
def comparar_dibujo(image_path, vertices, salida_lineas, salida_polygon):
    imagen_lineas = draw_vertices(image_path, vertices, display_text="objeto")
    imagen_lineas.save(salida_lineas)

    imagen_polygon = Image.open(image_path).copy()
    draw = ImageDraw.Draw(imagen_polygon)
    draw.polygon(vertices, outline="green")
    imagen_polygon.save(salida_polygon)

    return salida_lineas, salida_polygon


comparar_dibujo(
    "test.jpg",
    [(100, 100), (200, 100), (200, 200), (100, 200)],
    "con_lineas.jpg",
    "con_polygon.jpg",
)
```

</details>

## Ejercicio 3 — Combina OCR y detección de objetos con colores distintos

Completa `anotar_texto_y_objetos` para que, sobre la misma imagen, dibuje en un color los cuadros delimitadores de los objetos detectados (`object_localization`) y en otro color distinto los cuadros de las palabras detectadas (`text_detection`), reutilizando `draw_vertices` dos veces.

In [ ]:
def anotar_texto_y_objetos(image_path, output_path):
    pil_image = Image.open(image_path)
    width, height = pil_image.size

    with open(image_path, "rb") as image_file:
        content = image_file.read()
    image_vision = vision.Image(content=content)

    imagen_actual = pil_image

    # TODO 1: dibuja los objetos detectados en verde
    response_objetos = client.object_localization(image=image_vision)
    for obj in response_objetos.localized_object_annotations:
        vertices_px = ...
        imagen_actual = draw_vertices(imagen_actual, vertices_px, display_text=obj.name, color="green")

    # TODO 2: dibuja las palabras detectadas en azul (usa text_annotations[1:])
    response_texto = client.text_detection(image=image_vision)
    for palabra in response_texto.text_annotations[1:]:
        vertices_px = ...
        imagen_actual = draw_vertices(imagen_actual, vertices_px, color="blue")

    imagen_actual.save(output_path)
    return output_path


# anotar_texto_y_objetos("test.jpg", "test_anotado.jpg")


<details>
<summary><strong>Ver solución</strong></summary>

```python
def anotar_texto_y_objetos(image_path, output_path):
    pil_image = Image.open(image_path)
    width, height = pil_image.size

    with open(image_path, "rb") as image_file:
        content = image_file.read()
    image_vision = vision.Image(content=content)

    imagen_actual = pil_image

    response_objetos = client.object_localization(image=image_vision)
    for obj in response_objetos.localized_object_annotations:
        vertices_px = [(v.x * width, v.y * height) for v in obj.bounding_poly.normalized_vertices]
        imagen_actual = draw_vertices(imagen_actual, vertices_px, display_text=obj.name, color="green")

    response_texto = client.text_detection(image=image_vision)
    for palabra in response_texto.text_annotations[1:]:
        vertices_px = [(v.x, v.y) for v in palabra.bounding_poly.vertices]
        imagen_actual = draw_vertices(imagen_actual, vertices_px, color="blue")

    imagen_actual.save(output_path)
    return output_path


anotar_texto_y_objetos("test.jpg", "test_anotado.jpg")
```

</details>

---

## ¡Enhorabuena!

Has completado el recorrido práctico por las ocho lecciones del curso EITC/AI/GVAPI sobre la Google Vision API. Repasa el manual *Google Vision API for Dummies* siempre que necesites refrescar un concepto, y vuelve a estos notebooks para seguir practicando con tus propias imágenes antes del examen de certificación.